# RetSpike-Net
Hybrid spiking convolutional neural network for retinal diabetic retinopathy grading (APTOS 2019, 5-class).

## Architecture overview
Zone 1 - STDP encoder (Conv1 + Pool1)
- Unsupervised, label-blind. Learns DoG-compatible edge and contrast detectors from spike timing. Frozen after STDP pretraining.
- Filters: 32 channels, 5x5.

Zone 2 - Surrogate-gradient blocks (Conv2+Pool2, Conv3+Pool3)
- Supervised via backprop with fast-sigmoid surrogate.
- BatchNorm + Dropout for regularization.
- Filters: 64 channels (5x5) -> 128 channels (3x3).

Zone 3 - Classifier head
- Temporal feature fusion (mean + max + first-spike over T).
- Optional PCA(512) for feature compression before a linear classifier.
- Linear layer with weighted cross-entropy loss.

## Expected performance
- Optimistic: 82-85% weighted F1
- Realistic: 75-80% weighted F1
- Minimum bar: >70% weighted F1 (improvement over pure STDP ~65%)

## Dependencies
pip install snntorch torch torchvision scikit-learn numpy pandas tqdm matplotlib

## Optional installs
Run this cell only if your environment is missing the required packages.

In [1]:
# Uncomment if you need to install dependencies
# %pip install -q snntorch torch torchvision scikit-learn numpy pandas tqdm matplotlib

In [2]:
from pathlib import Path
import json
import random
import time

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import snntorch as snn
from snntorch import surrogate
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_recall_fscore_support,
)
from sklearn.utils.class_weight import compute_class_weight
from tqdm import tqdm
import matplotlib.pyplot as plt

In [3]:
SEED = 42
DATA_PROP = 1.0
NB_TIMESTEPS = 15
IMAGE_SIZE = 64
THRESHOLD = 15
STDP_EPOCHS = [5]
SG_EPOCHS = 30
BATCH_SIZE = 32
LR = 5e-4
LR_MIN = 1e-5
WEIGHT_DECAY = 1e-4
DROPOUT = 0.2
BETA = 0.9

CLASS_NAMES = ["No DR", "Mild", "Moderate", "Severe", "Prolif."]

NOTEBOOK_DIR = Path.cwd()
RETINAL_ROOT = NOTEBOOK_DIR.parent
SCNN_DIR = RETINAL_ROOT / "proto k-shot SCNN"
OUTPUT_DIR = NOTEBOOK_DIR / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CHECKPOINT_PATH = OUTPUT_DIR / "retspikenet_best.pth"
HISTORY_PATH = OUTPUT_DIR / "retspikenet_history.csv"
SPLIT_METRICS_PATH = OUTPUT_DIR / "retspikenet_split_metrics.csv"
RUN_SUMMARY_PATH = OUTPUT_DIR / "retspikenet_run_summary.json"

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Output dir:", OUTPUT_DIR)
print("Device:", DEVICE)

RUN_START = time.time()
RUN_TIMES = {}

Output dir: /media/aejaz/New Volume/Projects/SNN/Retinal Classification/RetSpike-Net/outputs
Device: cuda


In [4]:
def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


set_seed(SEED)

In [5]:
import sys

if not SCNN_DIR.exists():
    raise FileNotFoundError(f"Missing SCNN utilities at: {SCNN_DIR}")
sys.path.append(str(SCNN_DIR))

from utils import load_encoded_retinal_dataset
from snn import SpikingConv, SpikingPool, train_snn, SNN as STDPSNN

## LIF helper

In [6]:
def make_lif(beta: float = 0.9, threshold: float = 1.0) -> snn.Leaky:
    """Return a Leaky Integrate-and-Fire neuron with fast-sigmoid surrogate."""
    return snn.Leaky(
        beta=beta,
        threshold=threshold,
        spike_grad=surrogate.fast_sigmoid(slope=25),
        learn_beta=True,
        reset_mechanism="subtract",
    )

## Zone 2 - surrogate-gradient conv block

In [7]:
class SGConvBlock(nn.Module):
    """
    Surrogate-gradient spiking conv block:
      Conv2d -> BatchNorm -> LIF -> MaxPool -> Dropout

    Input/output shapes are in the spatial domain only; the timestep
    loop is handled by the outer RetSpikeNet forward() method.
    """

    def __init__(
        self,
        in_channels: int,
        out_channels: int,
        kernel_size: int,
        padding: int,
        pool: bool = True,
        dropout: float = 0.2,
        beta: float = 0.9,
    ):
        super().__init__()
        self.conv = nn.Conv2d(
            in_channels, out_channels, kernel_size,
            padding=padding, bias=False
        )
        self.bn = nn.BatchNorm2d(out_channels)
        self.lif = make_lif(beta=beta)
        self.pool = nn.MaxPool2d(2, 2) if pool else nn.Identity()
        self.drop = nn.Dropout2d(p=dropout)

    def init_mem(self, batch_size: int, channels: int, h: int, w: int,
                 device: torch.device) -> torch.Tensor:
        return torch.zeros(batch_size, channels, h, w, device=device)

    def forward(self, x: torch.Tensor, mem: torch.Tensor):
        """
        x   : (B, C_in, H, W) - one timestep input spikes
        mem : (B, C_out, H_out, W_out) - LIF membrane state

        Returns spikes, updated mem.
        """
        cur = self.drop(self.bn(self.conv(x)))
        spk, mem = self.lif(cur, mem)
        spk = self.pool(spk)
        return spk, mem

## Zone 1 - STDP encoder wrapper

In [8]:
class STDPEncoder:
    """
    Thin wrapper around the STDP SpikingConv/SpikingPool pair.
    After STDP pretraining, weights are frozen and this is used for inference.

    Input:  (T, C, H, W) numpy uint8 spike tensor for one sample
    Output: (T, 32, 32, 32) numpy spike tensor (after Conv1+Pool1)
    """

    def __init__(self, conv1: SpikingConv, pool1: SpikingPool):
        self.conv1 = conv1
        self.pool1 = pool1
        self.conv1.plasticity = False

    def __call__(self, x: np.ndarray) -> np.ndarray:
        """x shape: (T, C, H, W)"""
        T = x.shape[0]
        out_shape = (T,) + self.pool1.output_shape
        out = np.zeros(out_shape, dtype=np.float32)
        self.conv1.reset()
        self.pool1.reset()
        for t in range(T):
            spk = self.conv1(x[t].astype(np.float64), train=False)
            out[t] = self.pool1(spk)
        return out

## RetSpike-Net model

In [9]:
class RetSpikeNet(nn.Module):
    """
    Hybrid STDP + surrogate-gradient SCNN for retinal DR grading.

    Zone 1 (STDP, frozen): Conv1 32ch 5x5 -> Pool1 2x2 -> (32, 32, 32)
    Zone 2 (surrogate-gradient):
        Block2 64ch 5x5 + BN + LIF + Pool + Dropout -> (64, 16, 16)
        Block3 128ch 3x3 + BN + LIF + Pool + Dropout -> (128, 8, 8)
    Zone 3 (classifier):
        Temporal fusion (mean + max + first-spike) over T
        -> flat vector 3 * 128 * 8 * 8 = 24,576 dims
        -> Linear -> 5 classes
    """

    def __init__(
        self,
        nb_timesteps: int = 15,
        n_classes: int = 5,
        dropout: float = 0.2,
        beta: float = 0.9,
    ):
        super().__init__()
        self.T = nb_timesteps
        self.n_classes = n_classes

        self.block2 = SGConvBlock(32, 64, kernel_size=5, padding=2,
                                  pool=True, dropout=dropout, beta=beta)
        self.block3 = SGConvBlock(64, 128, kernel_size=3, padding=1,
                                  pool=True, dropout=dropout, beta=beta)

        self.classifier = nn.Linear(3 * 128 * 8 * 8, n_classes)

        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, nonlinearity="relu")
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.ones_(m.weight)
                nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                nn.init.zeros_(m.bias)

    def _compute_spk3_stack(self, x: torch.Tensor) -> torch.Tensor:
        B, T, _, H, W = x.shape
        device = x.device

        h2 = self.block2.init_mem(B, self.block2.conv.out_channels, H, W, device)
        h3 = self.block3.init_mem(B, self.block3.conv.out_channels, H // 2, W // 2, device)

        spk3_rec = []
        for t in range(T):
            xt = x[:, t]
            spk2, h2 = self.block2(xt, h2)
            spk3, h3 = self.block3(spk2, h3)
            spk3_rec.append(spk3)

        return torch.stack(spk3_rec, dim=0).permute(1, 0, 2, 3, 4)

    def _fuse_features(self, spk3_stack: torch.Tensor) -> torch.Tensor:
        T = spk3_stack.size(1)
        feat_mean = spk3_stack.mean(dim=1)
        feat_max = spk3_stack.max(dim=1).values
        first_mask = (spk3_stack.cumsum(dim=1) == 1) & (spk3_stack == 1)
        feat_first = first_mask.float().argmax(dim=1) / (T + 1e-8)

        return torch.cat([
            feat_mean.flatten(1),
            feat_max.flatten(1),
            feat_first.flatten(1),
        ], dim=1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        x : (B, T, C_in, H, W) - pre-encoded STDP spike tensors
            where C_in=32, H=32, W=32 (output of Zone 1)

        Returns logits: (B, n_classes)
        """
        spk3_stack = self._compute_spk3_stack(x)
        feat = self._fuse_features(spk3_stack)
        return self.classifier(feat)

    def forward_with_spikes(self, x: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        """Return logits and the spiking tensor for spike statistics."""
        spk3_stack = self._compute_spk3_stack(x)
        feat = self._fuse_features(spk3_stack)
        logits = self.classifier(feat)
        return logits, spk3_stack

## Training utilities

In [10]:
def compute_class_weights(y: np.ndarray, n_classes: int,
                          device: torch.device) -> torch.Tensor:
    """Inverse-frequency weights for cross-entropy loss."""
    classes = np.arange(n_classes)
    weights = compute_class_weight("balanced", classes=classes, y=y)
    return torch.tensor(weights, dtype=torch.float32, device=device)


def encode_zone1_batch(encoder: STDPEncoder,
                       X: np.ndarray,
                       desc: str = "Zone-1 encode") -> torch.Tensor:
    """
    Run the STDP encoder on the full dataset and return a tensor.

    X      : (N, T, C, H, W) uint8
    Returns: (N, T, 32, 32, 32) float32 tensor
    """
    encoded = []
    for x in tqdm(X, desc=desc):
        encoded.append(encoder(x))
    stacked = np.stack(encoded)
    return torch.from_numpy(stacked).float()

In [11]:
def train_retspikenet(
    model: RetSpikeNet,
    X_train_z1: torch.Tensor,
    y_train: np.ndarray,
    X_val_z1: torch.Tensor,
    y_val: np.ndarray,
    n_epochs: int = 30,
    batch_size: int = 32,
    lr: float = 1e-3,
    lr_min: float = 1e-5,
    weight_decay: float = 1e-4,
    device: torch.device = torch.device("cpu"),
    save_path: str = "retspikenet_best.pt",
) -> dict:
    """
    Train Zone 2 + Zone 3 with surrogate-gradient backprop.
    Zone 1 (STDP) inputs are already pre-encoded and frozen.

    Returns history dict with train/val loss and accuracy per epoch.
    """
    model = model.to(device)

    class_weights = compute_class_weights(y_train, model.n_classes, device)
    criterion = nn.CrossEntropyLoss(weight=class_weights)

    optimizer = torch.optim.AdamW(
        model.parameters(), lr=lr, weight_decay=weight_decay
    )
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=n_epochs, eta_min=lr_min
    )

    y_train_t = torch.tensor(y_train, dtype=torch.long)
    y_val_t = torch.tensor(y_val, dtype=torch.long)

    N = len(y_train_t)
    history = {
        "train_loss": [],
        "train_acc": [],
        "val_loss": [],
        "val_acc": [],
        "val_f1": [],
    }
    best_val_f1 = 0.0

    for epoch in range(1, n_epochs + 1):
        model.train()
        perm = torch.randperm(N)
        epoch_loss, epoch_correct = 0.0, 0

        for start in range(0, N, batch_size):
            idx = perm[start: start + batch_size]
            xb = X_train_z1[idx].to(device)
            yb = y_train_t[idx].to(device)

            optimizer.zero_grad()
            logits = model(xb)
            loss = criterion(logits, yb)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

            epoch_loss += loss.item() * len(idx)
            epoch_correct += (logits.argmax(1) == yb).sum().item()

        scheduler.step()

        train_loss = epoch_loss / N
        train_acc = epoch_correct / N

        model.eval()
        val_preds, val_loss_total = [], 0.0
        with torch.no_grad():
            for start in range(0, len(y_val_t), batch_size):
                xb = X_val_z1[start: start + batch_size].to(device)
                yb = y_val_t[start: start + batch_size].to(device)
                lg = model(xb)
                val_loss_total += criterion(lg, yb).item() * len(yb)
                val_preds.extend(lg.argmax(1).cpu().numpy())

        val_loss = val_loss_total / len(y_val_t)
        val_acc = accuracy_score(y_val, val_preds)
        val_f1 = f1_score(y_val, val_preds, average="weighted", zero_division=0)

        history["train_loss"].append(train_loss)
        history["train_acc"].append(train_acc)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)
        history["val_f1"].append(val_f1)

        print(
            f"Epoch {epoch:03d} | "
            f"train loss {train_loss:.4f} acc {train_acc:.3f} | "
            f"val loss {val_loss:.4f} acc {val_acc:.3f} "
            f"wF1 {val_f1:.3f}"
        )

        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            torch.save(model.state_dict(), save_path)
            print(f"  [checkpoint] best val wF1 = {best_val_f1:.4f}")

    print(f"\nTraining done. Best val wF1 = {best_val_f1:.4f}")
    return history

In [12]:
def evaluate_retspikenet(
    model: RetSpikeNet,
    X_z1: torch.Tensor,
    y_true: np.ndarray,
    batch_size: int = 32,
    device: torch.device = torch.device("cpu"),
    split_name: str = "test",
    compute_spike_metrics: bool = True,
) -> dict:
    """Full evaluation with class metrics and optional spike statistics."""
    model.eval().to(device)
    preds = []
    spike_counts = []
    spike_rates = []

    with torch.no_grad():
        for start in range(0, len(y_true), batch_size):
            xb = X_z1[start: start + batch_size].to(device)
            if compute_spike_metrics:
                logits, spk3_stack = model.forward_with_spikes(xb)
                total_spikes = spk3_stack.sum(dim=(1, 2, 3, 4))
                denom = float(
                    spk3_stack.shape[1]
                    * spk3_stack.shape[2]
                    * spk3_stack.shape[3]
                    * spk3_stack.shape[4]
                )
                spike_rate = total_spikes / max(denom, 1.0)
                spike_counts.append(total_spikes.cpu().numpy())
                spike_rates.append(spike_rate.cpu().numpy())
            else:
                logits = model(xb)
            preds.extend(logits.argmax(1).cpu().numpy())

    preds = np.array(preds)

    if y_true.size:
        acc = accuracy_score(y_true, preds)
        precision, recall, f1_weighted, _ = precision_recall_fscore_support(
            y_true,
            preds,
            average="weighted",
            zero_division=0,
        )
        wf1 = f1_score(y_true, preds, average="weighted", zero_division=0)
        mf1 = f1_score(y_true, preds, average="macro", zero_division=0)
    else:
        acc = 0.0
        precision = 0.0
        recall = 0.0
        f1_weighted = 0.0
        wf1 = 0.0
        mf1 = 0.0

    if compute_spike_metrics and spike_counts:
        spike_counts = np.concatenate(spike_counts)
        spike_rates = np.concatenate(spike_rates)
        mean_spikes = float(np.mean(spike_counts))
        std_spikes = float(np.std(spike_counts))
        mean_spike_rate = float(np.mean(spike_rates))
    else:
        mean_spikes = 0.0
        std_spikes = 0.0
        mean_spike_rate = 0.0

    print(f"\n=== {split_name.upper()} ===")
    print(f"Accuracy        : {acc:.4f}")
    print(f"Precision       : {precision:.4f}")
    print(f"Recall          : {recall:.4f}")
    print(f"F1 (weighted)   : {float(f1_weighted):.4f}")
    print(f"Macro F1        : {mf1:.4f}")
    if compute_spike_metrics:
        print(f"Mean spikes     : {mean_spikes:.2f}")
        print(f"Std spikes      : {std_spikes:.2f}")
        print(f"Mean spike rate : {mean_spike_rate:.6f}")

    return {
        "accuracy": float(acc),
        "precision": float(precision),
        "recall": float(recall),
        "f1_score": float(f1_weighted),
        "weighted_f1": float(wf1),
        "macro_f1": float(mf1),
        "mean_spikes": mean_spikes,
        "std_spikes": std_spikes,
        "mean_spike_rate": mean_spike_rate,
        "confusion": confusion_matrix(y_true, preds) if y_true.size else np.zeros((0, 0)),
        "preds": preds,
    }

## Pipeline - data loading

In [13]:
print("Loading dataset...")
t0 = time.time()
X_train, y_train, X_test, y_test = load_encoded_retinal_dataset(
    data_prop=DATA_PROP,
    nb_timesteps=NB_TIMESTEPS,
    image_size=IMAGE_SIZE,
    threshold=THRESHOLD,
    oversample=True,
    oversample_strategy="oversample",
    augment_minority=True,
    seed=SEED,
)
RUN_TIMES["load_data_sec"] = time.time() - t0

N = len(y_train)
val_size = int(N * 0.2)
rng = np.random.default_rng(SEED)
perm = rng.permutation(N)
val_idx = perm[:val_size]
train_idx = perm[val_size:]

X_val, y_val = X_train[val_idx], y_train[val_idx]
X_train, y_train = X_train[train_idx], y_train[train_idx]

print(f"Train: {X_train.shape}  Val: {X_val.shape}  Test: {X_test.shape}")
print(f"Data load elapsed (min): {RUN_TIMES['load_data_sec'] / 60:.2f}")

Loading dataset...
[utils] Class distribution before balancing: [(0, 1444), (1, 296), (2, 799), (3, 154), (4, 236)]
[utils] Class distribution after  balancing: [(0, 1444), (1, 1444), (2, 1444), (3, 1444), (4, 1444)]
[utils] Encoding train spikes …
[utils] Encoding test  spikes …
Train: (5776, 15, 2, 64, 64)  Val: (1444, 15, 2, 64, 64)  Test: (733, 15, 2, 64, 64)
Data load elapsed (min): 3.55


## Zone 1 - STDP pretraining

In [14]:
print("Zone 1 STDP pretraining...")
t0 = time.time()
input_shape = X_train[0][0].shape
stdp_net = STDPSNN(input_shape)

train_snn(
    stdp_net,
    X_train,
    y_train,
    epochs=STDP_EPOCHS,
    convergence_threshold=0.0,
    seed=SEED,
)

encoder = STDPEncoder(stdp_net.conv_layers[0], stdp_net.pool_layers[0])
RUN_TIMES["stdp_train_sec"] = time.time() - t0
print(f"STDP elapsed (min): {RUN_TIMES['stdp_train_sec'] / 60:.2f}")

Zone 1 STDP pretraining...

[STDP] Training Conv Layer 1 / 3  (5 epoch(s), convergence_threshold=0.0)


  Layer 1 epoch 1/5: 100%|██████████| 5776/5776 [15:35<00:00,  6.18it/s]


  epoch 1 done | convergence=0.000045


  Layer 1 epoch 2/5: 100%|██████████| 5776/5776 [13:34<00:00,  7.09it/s]


  epoch 2 done | convergence=0.000000


...

  epoch 3 done | convergence=0.000007


  Layer 3 epoch 4/5: 100%|██████████| 5776/5776 [27:09<00:00,  3.54it/s]


  epoch 4 done | convergence=0.000000


  Layer 3 epoch 5/5: 100%|██████████| 5776/5776 [19:21<00:00,  4.97it/s]

  epoch 5 done | convergence=0.000006
STDP elapsed (min): 327.76


## Zone 1 encoding (frozen)

In [15]:
print("Encoding splits through frozen Zone 1...")
t0 = time.time()
X_train_z1 = encode_zone1_batch(encoder, X_train, desc="Train zone-1")
X_val_z1 = encode_zone1_batch(encoder, X_val, desc="Val zone-1")
X_test_z1 = encode_zone1_batch(encoder, X_test, desc="Test zone-1")
RUN_TIMES["zone1_encode_sec"] = time.time() - t0

print("Encoded shapes:", X_train_z1.shape, X_val_z1.shape, X_test_z1.shape)
print(f"Zone-1 encode elapsed (min): {RUN_TIMES['zone1_encode_sec'] / 60:.2f}")

Encoding splits through frozen Zone 1...


Test zone-1: 100%|██████████| 733/733 [00:19<00:00, 36.81it/s]


Encoded shapes: torch.Size([5776, 15, 32, 32, 32]) torch.Size([1444, 15, 32, 32, 32]) torch.Size([733, 15, 32, 32, 32])
Zone-1 encode elapsed (min): 3.97


## Train RetSpike-Net (Zone 2 + Zone 3)

In [16]:
model = RetSpikeNet(
    nb_timesteps=NB_TIMESTEPS,
    n_classes=len(CLASS_NAMES),
    dropout=DROPOUT,
    beta=BETA,
)

param_count = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable parameters: {param_count:,}")

train_start = time.time()
train_start_str = time.strftime("%Y-%m-%d %H:%M:%S", time.localtime(train_start))
print(f"SG training start: {train_start_str}")

history = train_retspikenet(
    model,
    X_train_z1,
    y_train,
    X_val_z1,
    y_val,
    n_epochs=SG_EPOCHS,
    batch_size=BATCH_SIZE,
    lr=LR,
    lr_min=LR_MIN,
    weight_decay=WEIGHT_DECAY,
    device=DEVICE,
    save_path=str(CHECKPOINT_PATH),
)
RUN_TIMES["sg_train_sec"] = time.time() - train_start
print(f"SG training elapsed (min): {RUN_TIMES['sg_train_sec'] / 60:.2f}")

Trainable parameters: 248,199
SG training start: 2026-05-07 07:13:01
Epoch 001 | train loss 1.7361 acc 0.411 | val loss 1.5421 acc 0.394 wF1 0.346
  [checkpoint] best val wF1 = 0.3457
Epoch 002 | train loss 1.4309 acc 0.472 | val loss 1.6318 acc 0.493 wF1 0.437
  [checkpoint] best val wF1 = 0.4373
Epoch 003 | train loss 1.2696 acc 0.516 | val loss 1.6406 acc 0.403 wF1 0.355
Epoch 004 | train loss 1.2725 acc 0.524 | val loss 1.4587 acc 0.494 wF1 0.445
  [checkpoint] best val wF1 = 0.4451
Epoch 005 | train loss 1.2173 acc 0.539 | val loss 1.3221 acc 0.497 wF1 0.454
  [checkpoint] best val wF1 = 0.4540
Epoch 006 | train loss 1.1581 acc 0.557 | val loss 1.2604 acc 0.512 wF1 0.497
  [checkpoint] best val wF1 = 0.4968
Epoch 007 | train loss 1.1025 acc 0.569 | val loss 1.1617 acc 0.550 wF1 0.545
  [checkpoint] best val wF1 = 0.5455
Epoch 008 | train loss 1.1801 acc 0.566 | val loss 1.2435 acc 0.528 wF1 0.510
Epoch 009 | train loss 1.0974 acc 0.593 | val loss 1.6962 acc 0.496 wF1 0.436
Epoch 0

## Save training history

In [17]:
history_df = pd.DataFrame(history)
history_df.to_csv(HISTORY_PATH, index=False)
print("Saved history to:", HISTORY_PATH)
print("Best checkpoint:", CHECKPOINT_PATH)

history_df.head()

Saved history to: /media/aejaz/New Volume/Projects/SNN/Retinal Classification/RetSpike-Net/outputs/retspikenet_history.csv
Best checkpoint: /media/aejaz/New Volume/Projects/SNN/Retinal Classification/RetSpike-Net/outputs/retspikenet_best.pth


,train_loss,train_acc,val_loss,val_acc,val_f1
0,1.736125,0.410665,1.542117,0.394044,0.345677
1,1.430864,0.471780,1.631795,0.493075,0.437319
2,1.269554,0.516274,1.640597,0.403047,0.355244
3,1.272485,0.524238,1.458692,0.493767,0.445108
4,1.217327,0.538608,1.322145,0.497230,0.454021


## Evaluate and save reports

In [18]:
def save_split_reports(split_name: str, y_true: np.ndarray, y_pred: np.ndarray,
                       output_dir: Path, class_names: list[str]) -> None:
    report = classification_report(
        y_true,
        y_pred,
        target_names=class_names,
        zero_division=0,
        output_dict=True,
    )
    report_df = pd.DataFrame(report).transpose()
    report_path = output_dir / f"{split_name}_classification_report.csv"
    report_df.to_csv(report_path)

    cm = confusion_matrix(y_true, y_pred, labels=list(range(len(class_names))))
    cm_df = pd.DataFrame(cm, index=class_names, columns=class_names)
    cm_path = output_dir / f"{split_name}_confusion_matrix.csv"
    cm_df.to_csv(cm_path)

    fig, ax = plt.subplots(figsize=(6, 5))
    im = ax.imshow(cm, interpolation="nearest", cmap="Blues")
    ax.figure.colorbar(im, ax=ax)
    ax.set(
        xticks=np.arange(len(class_names)),
        yticks=np.arange(len(class_names)),
        xticklabels=class_names,
        yticklabels=class_names,
        ylabel="True label",
        xlabel="Predicted label",
        title=f"Confusion Matrix - {split_name}",
    )
    plt.setp(ax.get_xticklabels(), rotation=45, ha="right", rotation_mode="anchor")

    threshold = cm.max() / 2.0 if cm.size else 0.0
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(
                j,
                i,
                format(cm[i, j], "d"),
                ha="center",
                va="center",
                color="white" if cm[i, j] > threshold else "black",
            )

    plt.tight_layout()
    cm_img_path = output_dir / f"{split_name}_confusion_matrix.png"
    plt.savefig(cm_img_path, dpi=200)
    plt.close(fig)

    print("Saved report to:", report_path)
    print("Saved confusion matrix to:", cm_path)
    print("Saved confusion matrix image to:", cm_img_path)


if CHECKPOINT_PATH.exists():
    model.load_state_dict(torch.load(CHECKPOINT_PATH, map_location=DEVICE))

split_data = {
    "train": (X_train_z1, y_train),
    "val": (X_val_z1, y_val),
    "test": (X_test_z1, y_test),
}

metrics_rows = []

for split_name, (X_z1, y_true) in split_data.items():
    metrics = evaluate_retspikenet(
        model,
        X_z1,
        y_true,
        batch_size=BATCH_SIZE,
        device=DEVICE,
        split_name=split_name,
        compute_spike_metrics=True,
    )
    metrics_rows.append({
        "split": split_name,
        "accuracy": metrics["accuracy"],
        "precision": metrics["precision"],
        "recall": metrics["recall"],
        "f1_score": metrics["f1_score"],
        "weighted_f1": metrics["weighted_f1"],
        "macro_f1": metrics["macro_f1"],
        "mean_spikes": metrics["mean_spikes"],
        "std_spikes": metrics["std_spikes"],
        "mean_spike_rate": metrics["mean_spike_rate"],
    })

    save_split_reports(split_name, y_true, metrics["preds"], OUTPUT_DIR, CLASS_NAMES)

metrics_df = pd.DataFrame(metrics_rows)
metrics_df.to_csv(SPLIT_METRICS_PATH, index=False)
print("Saved split metrics to:", SPLIT_METRICS_PATH)
metrics_df


=== TRAIN ===
Accuracy        : 0.8298
Precision       : 0.8309
Recall          : 0.8298
F1 (weighted)   : 0.8261
Macro F1        : 0.8257
Mean spikes     : 34092.70
Std spikes      : 5646.44
Mean spike rate : 0.277447
Saved report to: /media/aejaz/New Volume/Projects/SNN/Retinal Classification/RetSpike-Net/outputs/train_classification_report.csv
Saved confusion matrix to: /media/aejaz/New Volume/Projects/SNN/Retinal Classification/RetSpike-Net/outputs/train_confusion_matrix.csv
Saved confusion matrix image to: /media/aejaz/New Volume/Projects/SNN/Retinal Classification/RetSpike-Net/outputs/train_confusion_matrix.png

=== VAL ===
Accuracy        : 0.6496
Precision       : 0.6451
Recall          : 0.6496
F1 (weighted)   : 0.6413
Macro F1        : 0.6449
Mean spikes     : 34160.13
Std spikes      : 5557.79
Mean spike rate : 0.277996
Saved report to: /media/aejaz/New Volume/Projects/SNN/Retinal Classification/RetSpike-Net/outputs/val_classification_report.csv
Saved confusion matrix to: /

,split,accuracy,precision,recall,f1_score,weighted_f1,macro_f1,mean_spikes,std_spikes,mean_spike_rate
0,train,0.829813,0.830892,0.829813,0.826136,0.826136,0.825690,34092.699219,5646.443359,0.277447
1,val,0.649584,0.645115,0.649584,0.641252,0.641252,0.644908,34160.128906,5557.791016,0.277996
2,test,0.708049,0.699909,0.708049,0.701887,0.701887,0.498458,34628.371094,5518.895020,0.281806


## Optional PCA + logistic regression head

In [19]:
RUN_PCA_HEAD = False


def extract_features(model: RetSpikeNet, X_z1: torch.Tensor,
                     batch_size: int, device: torch.device) -> np.ndarray:
    model.eval().to(device)
    feats = []
    with torch.no_grad():
        for start in range(0, len(X_z1), batch_size):
            xb = X_z1[start: start + batch_size].to(device)
            B, T, C, H, W = xb.shape

            h2 = model.block2.init_mem(B, model.block2.conv.out_channels, H, W, device)
            h3 = model.block3.init_mem(B, model.block3.conv.out_channels, H // 2, W // 2, device)

            spk3_rec = []
            for t in range(T):
                xt = xb[:, t]
                spk2, h2 = model.block2(xt, h2)
                spk3, h3 = model.block3(spk2, h3)
                spk3_rec.append(spk3)

            spk3_stack = torch.stack(spk3_rec, dim=0).permute(1, 0, 2, 3, 4)
            feat_mean = spk3_stack.mean(dim=1)
            feat_max = spk3_stack.max(dim=1).values
            first_mask = (spk3_stack.cumsum(dim=1) == 1) & (spk3_stack == 1)
            feat_first = first_mask.float().argmax(dim=1) / (T + 1e-8)

            feat = torch.cat(
                [feat_mean.flatten(1), feat_max.flatten(1), feat_first.flatten(1)],
                dim=1,
            )
            feats.append(feat.cpu().numpy())

    return np.concatenate(feats, axis=0)


if RUN_PCA_HEAD:
    print("Extracting features for PCA head...")
    train_feat = extract_features(model, X_train_z1, BATCH_SIZE, DEVICE)
    val_feat = extract_features(model, X_val_z1, BATCH_SIZE, DEVICE)

    pca = PCA(n_components=512, random_state=SEED)
    train_pca = pca.fit_transform(train_feat)
    val_pca = pca.transform(val_feat)

    clf = LogisticRegression(
        max_iter=2000,
        class_weight="balanced",
        n_jobs=-1,
    )
    clf.fit(train_pca, y_train)
    val_pred = clf.predict(val_pca)

    val_acc = accuracy_score(y_val, val_pred)
    val_wf1 = f1_score(y_val, val_pred, average="weighted", zero_division=0)
    print(f"PCA head val acc: {val_acc:.4f}")
    print(f"PCA head val wF1: {val_wf1:.4f}")

## Run summary

In [20]:
RUN_TIMES["total_sec"] = time.time() - RUN_START
run_summary = {
    "run_start": time.strftime("%Y-%m-%d %H:%M:%S", time.localtime(RUN_START)),
    "run_end": time.strftime("%Y-%m-%d %H:%M:%S", time.localtime(time.time())),
    "durations_sec": RUN_TIMES,
    "checkpoint": str(CHECKPOINT_PATH),
    "history_csv": str(HISTORY_PATH),
    "split_metrics_csv": str(SPLIT_METRICS_PATH),
}

with open(RUN_SUMMARY_PATH, "w", encoding="utf-8") as f:
    json.dump(run_summary, f, indent=2)

print("Total runtime (min):", RUN_TIMES["total_sec"] / 60.0)
print("Saved run summary to:", RUN_SUMMARY_PATH)

Total runtime (min): 339.97178717851637
Saved run summary to: /media/aejaz/New Volume/Projects/SNN/Retinal Classification/RetSpike-Net/outputs/retspikenet_run_summary.json
